# 03 · API card (harder): NASA POWER, a very hot day?

**NASA Space Apps Cairo 2026 · Bootcamp · API & Data Integration**

**Goal:** using 20 years of daily Cairo data, find the *chance of a very hot day* (max temperature above 35 °C) around a calendar date, and offer a CSV with units and the source.

**No key needed** · **Trap:** `-999` means *missing*, not −999 °C.

> ✍️ **Starter notebook.** Cells marked `TODO` are yours to finish. Green ✅ check cells tell you when a step works.

Open in Colab: *File → Save a copy in Drive* first, so your edits are kept.

## Setup
NASA POWER needs **no key**, so `API_KEY` is not used in this notebook.

In [ ]:
# ▶ Run this cell first
import os, json, requests
import pandas as pd
from datetime import date, timedelta

API_KEY = "DEMO_KEY"   # 👉 replace with YOUR free key from https://api.nasa.gov (DEMO_KEY runs out fast in a full room)
# Never paste a paid key (for example an AI key) into a notebook you share.

def check(test, good, hint):
    """Tiny self-check: prints ✅ when the step works, ❌ plus a hint when it does not."""
    try:
        ok = bool(test())
    except Exception:
        ok = False
    print(("✅ " + good) if ok else ("❌ " + hint))

def show_url(url, params):
    """Print a request URL with your key hidden, so it is safe to show on a projector."""
    req = requests.Request("GET", url, params=params).prepare()
    key = str(params.get("api_key", ""))
    print(req.url.replace(key, "YOUR_KEY") if key else req.url)

In [ ]:
# 🛟 Safety net: SYNTHETIC data in the shape of a NASA POWER answer. Made up by a formula, NOT real Cairo weather.
import numpy as np

def make_synthetic_power():
    days = pd.date_range("2005-01-01", "2024-12-31", freq="D")
    rng = np.random.default_rng(42)
    season = 29 + 9 * np.sin((days.dayofyear.values - 110) / 365.25 * 2 * np.pi)
    values = np.round(season + rng.normal(0, 2.5, len(days)), 2)
    values[rng.random(len(days)) < 0.01] = -999          # a few "missing" values, like the real data
    series = {d.strftime("%Y%m%d"): float(v) for d, v in zip(days, values)}
    return {"properties": {"parameter": {"T2M_MAX": series}}}

def sample_banner():
    print("⚠️  USING SYNTHETIC SAMPLE DATA, not real NASA data. Fix the call to see real results.")

## 1 · Build the request
Endpoint: `https://power.larc.nasa.gov/api/temporal/daily/point`

Parameters you need: `parameters` (the variable: **T2M_MAX** = daily maximum temperature at 2 m, °C), `community` (use `RE`), `longitude`, `latitude`, `start` and `end` (as **YYYYMMDD**), `format` (`JSON`).

Cairo is at about **30.0444 N, 31.2357 E**. Ask for 2005-01-01 to 2024-12-31.

### Your turn

In [ ]:
URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
CITY, LAT, LON = "Cairo", ..., ...        # TODO: latitude and longitude
params = {
    "parameters": ...,
    "community": ...,
    "longitude": LON,
    "latitude": LAT,
    "start": ...,       # TODO: YYYYMMDD
    "end": ...,
    "format": "JSON",
}
show_url(URL, params)

In [ ]:
check(lambda: params["parameters"] == "T2M_MAX" and len(params["start"]) == 8 and 29 < LAT < 31,
      "Request is ready. Test the printed URL in a browser tab.", "parameters should be T2M_MAX, dates 8 digits (YYYYMMDD), latitude about 30.")

## 2 · Make the call
This answer is large (about 7,300 days), so give it more time.

### Your turn

In [ ]:
resp = requests.get(..., params=..., timeout=60)
print("Status:", resp.status_code)
js = resp.json()
print(list(js.keys()))

In [ ]:
check(lambda: resp.status_code == 200 and "properties" in js, "200 OK.", "Not 200, or no 'properties' in the answer. Check the parameter names.")

## 3 · Find the data in the JSON and count the gaps
The temperatures are in `properties` → `parameter` → `T2M_MAX`: a dictionary of `"YYYYMMDD": value`. Make a table with a real date column and count how many values are `-999`.

### Your turn

In [ ]:
series = ...        # TODO: the T2M_MAX dictionary
df = pd.DataFrame({"date": list(series.keys()), "t2m_max": list(series.values())})
df["date"] = ...    # TODO: turn text like 20050101 into real dates (pd.to_datetime with format="%Y%m%d")
missing = ...       # TODO: how many values equal -999?
print(len(df), "days,", missing, "missing")

In [ ]:
check(lambda: str(df["date"].dtype).startswith("datetime") and len(df) > 7000, "Real dates, about 20 years.", "date must be a datetime and you should have ~7,300 rows.")

## 4 · Clean: remove the `-999` values
If you averaged them in, a 20-year chance would be badly wrong. Drop them and say how many you dropped.

### Your turn

In [ ]:
before = len(df)
df = ...            # TODO: keep only rows where t2m_max is not -999
print(f"Dropped {before - len(df)} missing days, {len(df)} valid days left")

In [ ]:
check(lambda: (df["t2m_max"] > -900).all(), "No -999 left.", "Some -999 values are still in your table.")

## 5 · Compute the chance of a very hot day
For a calendar date (say **15 July**) look at that day **± 3 days** in every year, and count how many of those days had a maximum above **35 °C**. With a window you get ~140 days of evidence instead of only 20.

<details><summary>Hint</summary>

Build, for each row, the same month/day in the row's own year, then keep rows whose date is within `WINDOW` days of it: `target = pd.to_datetime(pd.DataFrame({"year": df["date"].dt.year, "month": MONTH, "day": DAY}))`, then `(df["date"] - target).abs().dt.days <= WINDOW`.

</details>

### Your turn

In [ ]:
MONTH, DAY, WINDOW, THRESHOLD = 7, 15, 3, 35      # 15 July ± 3 days, above 35 °C

target = ...        # TODO: the same month/day in each row's year
near = ...          # TODO: True where the row is within WINDOW days of target
sel = df[near]
chance = ...        # TODO: percentage of days in sel with t2m_max above THRESHOLD
print(f"{CITY}, {DAY}/{MONTH} ± {WINDOW} days: {chance:.0f}% of days were above {THRESHOLD} °C ({len(sel)} valid days, 2005-2024)")

In [ ]:
check(lambda: 0 <= chance <= 100 and 100 < len(sel) < 200, "A sensible percentage from ~140 days.", "Expect between 100 and 200 days in your window and a chance between 0 and 100.")

## 6 · Offer a CSV download with units and the source
A file someone else can trust says **what** each column is, in **which units**, and **where** it came from.

### Your turn

In [ ]:
out = sel[["date", "t2m_max"]].rename(columns={"t2m_max": "max_temp_c"})
# TODO: add columns: units ("°C"), source ("NASA POWER daily point, T2M_MAX"), and a source_url
os.makedirs("backup", exist_ok=True)
out.to_csv("backup/cairo_hot_days.csv", index=False)
out.head()

In [ ]:
check(lambda: {"units", "source"} <= set(out.columns) and os.path.exists("backup/cairo_hot_days.csv"),
      "CSV with units and source saved.", "Add units and source columns, then save the CSV.")

## 7 · Plan for failure
POWER can be slow with big requests. Write `load_power(params)` that returns `(json, source)`: timeout → clear message → saved copy of the raw answer (`backup/power_raw.json`) → synthetic sample with the warning banner as the last resort. Save the raw answer whenever a live call works.

### Your turn

In [ ]:
def load_power(p):
    try:
        r = requests.get(URL, params=p, timeout=60)
        # TODO: clear message for non-200 answers
        data = r.json()
        # TODO: save data to backup/power_raw.json
        return data, "live"
    except (requests.RequestException, ValueError) as e:
        print("⚠️  API problem:", e)
        # TODO: saved copy if it exists, else sample_banner() + make_synthetic_power()
        ...

In [ ]:
_, source = load_power({**params, "parameters": "NOT_A_REAL_PARAMETER"})
check(lambda: source in ("saved copy", "sample"), "A bad request no longer breaks your feature.",
      "With a wrong parameter your function should print a message and return saved or sample data.")

## ✅ Done means
- [ ] a `200 OK` POWER call
- [ ] `-999` removed and counted
- [ ] the chance of a very hot day, with how many days it is based on
- [ ] a CSV with units and source
- [ ] one failure handled

**Stretch:** make `MONTH`, `DAY` and `CITY` inputs, or compare two cities.